In [93]:
import pandas as pd
t_and_a_df_ASC = pd.read_pickle("t&a_df_ASC.pk1")
t_and_a_df_MAIN = pd.read_pickle("t&a_df_MAIN.pk1")
t_and_a_df_KOP = pd.read_pickle("t&a_df_KOP.pk1")
t_and_a_df_BUCKS = pd.read_pickle("t&a_df_BUCKS.pk1")
t_and_a_df_VOORHEES = pd.read_pickle("t&a_df_VOORHEES.pk1")
t_and_a_df_BRANDYWINE = pd.read_pickle("t&a_df_BRANDYWINE.pk1")

In [122]:
def create_pain_distribution_df(df, pain_score_column):
    prevalence_df = df[["med_combos", pain_score_column]]

    bins = [-1, 3, 6, 10]
    labels = ["0-3", "4-6", "7-10"]

    prevalence_df["pain_bucket"] = pd.cut(prevalence_df[pain_score_column], bins=bins, labels=labels)

    percentages = pd.crosstab(prevalence_df["med_combos"], prevalence_df["pain_bucket"], normalize="index").mul(100).round(1)
    counts = pd.crosstab(prevalence_df["med_combos"], prevalence_df["pain_bucket"])

    table = counts.astype(str) + " (" + percentages.astype(str) + "%)"

    return percentages



In [ ]:
first_pacu_pain_score_dist_ASC = create_pain_distribution_df(t_and_a_df_ASC, "FIRST_PACU_PAIN_SCORE")
first_pacu_pain_score_dist_MAIN = create_pain_distribution_df(t_and_a_df_MAIN, "FIRST_PACU_PAIN_SCORE")
first_pacu_pain_score_dist_KOP = create_pain_distribution_df(t_and_a_df_KOP, "FIRST_PACU_PAIN_SCORE")
first_pacu_pain_score_dist_BUCKS = create_pain_distribution_df(t_and_a_df_BUCKS, "FIRST_PACU_PAIN_SCORE")
first_pacu_pain_score_dist_VOORHEES = create_pain_distribution_df(t_and_a_df_VOORHEES, "FIRST_PACU_PAIN_SCORE")
first_pacu_pain_score_dist_BRANDYWINE = create_pain_distribution_df(t_and_a_df_BRANDYWINE, "FIRST_PACU_PAIN_SCORE")

highest_pacu_pain_score_dist_MAIN = create_pain_distribution_df(t_and_a_df_MAIN, "HIGHEST_PACU_PAIN_SCORE")
highest_pacu_pain_score_dist_ASC = create_pain_distribution_df(t_and_a_df_ASC, "HIGHEST_PACU_PAIN_SCORE")
highest_pacu_pain_score_dist_KOP = create_pain_distribution_df(t_and_a_df_KOP, "HIGHEST_PACU_PAIN_SCORE")
highest_pacu_pain_score_dist_BUCKS = create_pain_distribution_df(t_and_a_df_BUCKS, "HIGHEST_PACU_PAIN_SCORE")
highest_pacu_pain_score_dist_VOORHEES = create_pain_distribution_df(t_and_a_df_VOORHEES, "HIGHEST_PACU_PAIN_SCORE")
highest_pacu_pain_score_dist_BRANDYWINE = create_pain_distribution_df(t_and_a_df_BRANDYWINE, "HIGHEST_PACU_PAIN_SCORE")


In [25]:
import matplotlib.pyplot as plt
def graph_pain_distribution(df):
    ax = df.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
    )

    ax.set_ylabel("Percentage of Patients")
    ax.set_xlabel("Medication Combination")
    ax.set_title("Pain Score Distribution by Medication Combination")

    plt.legend(title="Pain Score", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.xticks(rotation=45)

    plt.show()


In [ ]:
graph_pain_distribution(highest_pacu_pain_score_dist_BRANDYWINE)

In [38]:
"""
Run Kruskal-Wallis (because data is not normally distributed)
"""
from scipy.stats import kruskal

def perform_Kruskal_Wallis(df, pain_score):
    groups = [group[pain_score].astype(int).values for _, group in df.groupby('med_combos') if len(group) >= 2]

    f_stat, p_value = kruskal(*groups)
    for group in groups:
        if len(group) < 2:
            print(group)

    print("F-statistic:", f_stat)
    print("p-value:", p_value)
    print("--------------------")

In [137]:
"""
Run post-hoc DUNN test for pain scores in medication combinations
p-values adjusted for multiple comparisons using bonferroni
"""
import scikit_posthocs as sp

def perform_posthoc(df, pain_score):
    result = sp.posthoc_dunn(df, val_col=pain_score, group_col='med_combos', p_adjust='bonferroni')
    formatted_pvals = result.applymap(lambda x: f"{x:.8f}")
    significant_results = []
    all_results = []

    for i in range(len(result)):
        for j in range(i + 1, len(result)):
            p_val = formatted_pvals.iloc[i,j]
            if float(p_val) < 0.05:
                significant_results.append((result.index[i], result.columns[j], p_val))
            all_results.append((result.index[i], result.columns[j], p_val))


    significant_results_df = pd.DataFrame(significant_results, columns=['Group1', 'Group2', 'p_value'])
    all_results_df = pd.DataFrame(all_results, columns=['Group1', 'Group2', 'p_value'])
    print("significant results:", significant_results_df)
    print("all results:", all_results_df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=t_and_a_df_ASC, x="LOCATION", y="HIGHEST_PACU_PAIN_SCORE")
#sns.swarmplot(data=t_and_a_df_ASC, x="LOCATION", y="HIGHEST_PACU_PAIN_SCORE", color="black", alpha=0.3)

plt.xticks(rotation=45)
plt.title("Highest Pain Score by Surgery Center")
plt.show()

In [ ]:
counts = pd.crosstab(t_and_a_df_ASC["LOCATION"], t_and_a_df_ASC["med_combos"])

percentages = pd.crosstab(
    t_and_a_df_ASC["LOCATION"],
    t_and_a_df_ASC["med_combos"],
    normalize="index"   # percent within each surgery center
) * 100

table = table.round(1)
summary = counts.astype(str) + " (" + percentages.round(1).astype(str) + "%)"# nicer formatting

In [ ]:
summary = (
    t_and_a_df_ASC.groupby("LOCATION")["HIGHEST_PACU_PAIN_SCORE"]
      .agg(
          mean="mean",
          q1=lambda x: x.quantile(0.25),
          median="median",
          q3=lambda x: x.quantile(0.75)
      )
)

summary["IQR"] = summary["q3"] - summary["q1"]
summary = summary.round(2)

In [ ]:
groups = [
    t_and_a_df_ASC[t_and_a_df_ASC["LOCATION"] == "VOORHEES DAY SURGERY"]["HIGHEST_PACU_PAIN_SCORE"],
    t_and_a_df_ASC[t_and_a_df_ASC["LOCATION"] == "BUCKS DAY SURGERY"]["HIGHEST_PACU_PAIN_SCORE"],
    t_and_a_df_ASC[t_and_a_df_ASC["LOCATION"] == "BRANDYWINE VALLEY DAY SURGERY"]["HIGHEST_PACU_PAIN_SCORE"],
]

kruskal(*groups)

In [166]:
"""
Run post-hoc DUNN test for pain scores between surgery centers
"""
import scikit_posthocs as sp

def perform_posthoc_surgery_centers(df, pain_score):
    result = sp.posthoc_dunn(df, val_col=pain_score, group_col='LOCATION', p_adjust='bonferroni')
    formatted_pvals = result.applymap(lambda x: f"{x:.2e}")
    significant_results = []
    all_results = []

    for i in range(len(result)):
        for j in range(i + 1, len(result)):
            p_val = formatted_pvals.iloc[i,j]
            if float(p_val) < 0.05:
                significant_results.append((result.index[i], result.columns[j], p_val))
            all_results.append((result.index[i], result.columns[j], p_val))


    significant_results_df = pd.DataFrame(significant_results, columns=['Group1', 'Group2', 'p_value'])
    all_results_df = pd.DataFrame(all_results, columns=['Group1', 'Group2', 'p_value'])
    print("significant results:", significant_results_df)
    print("all results:", all_results_df)